# Laminated plate and ABD matrix

`laminated_plate` creates a `Laminate` from the stacking sequence, from the
bottom to the top ply, see [Conventions](../conventions.rst). The `ABD` matrix
of the CLPT and the FSDT, the corrected transverse shear stiffness `Ats`, the
equivalent properties and the mass integrals are computed at once. The code
below is the unit test `test_example_laminated_plate` of `tests/test_core.py`,
followed by the test `test_laminated_plate_plane_stress` of the same file.

## What the first test verifies

The quasi-isotropic laminate $[0/45/-45/90]_s$ is checked against properties
that follow from the theory, not against stored numbers. The stiffnesses are
the thickness integrals of the ply stiffnesses $C^{(k)}$ in the laminate axes,
see [Laminate stiffness and plate theories](../constitutive_matrices.rst):

$$
(A, B, D) = \sum_k \int_{z_k}^{z_{k+1}} C^{(k)} (1, z, z^2) \, dz, \qquad
\begin{Bmatrix} N \\ M \end{Bmatrix} =
\begin{bmatrix} A & B \\ B & D \end{bmatrix}
\begin{Bmatrix} \varepsilon^{(0)} \\ \varepsilon^{(1)} \end{Bmatrix}
$$

- **Layout of `ABD`**: the blocks of `lam.ABD` are `lam.A`, `lam.B` and
  `lam.D`, in the order $\{N_{xx}, N_{yy}, N_{xy}, M_{xx}, M_{yy}, M_{xy}\}$.
- **No membrane-bending coupling**: the laminate is symmetric about the
  mid-surface and has no offset, so the integrand $C^{(k)} z$ is odd about
  $z = 0$ and $B = 0$, to round-off relative to $A_{11} h$.
- **Quasi-isotropic membrane stiffness**: the ply stiffnesses are linear in
  $\cos 2\theta$, $\sin 2\theta$, $\cos 4\theta$, $\sin 4\theta$ with the
  material invariants $u_1, \dots, u_5$ as coefficients, see
  [Ply constitutive relations and invariants](../lamina.rst). For the angles
  0, 45, -45 and 90 degrees, each in the same thickness, the averages of the
  four trigonometric terms, the lamination parameters $\xi_{A1}, \dots,
  \xi_{A4}$, are zero, and the stiffnesses of
  [Lamination parameters](../lamination_parameters.rst) reduce to
  $A_{11} = A_{22} = h u_1$, $A_{12} = h u_4$, $A_{66} = h u_5$ and
  $A_{16} = A_{26} = 0$. With $u_5 = (u_1 - u_4)/2$ this is
  $A_{66} = (A_{11} - A_{12})/2$, the form of the membrane stiffness of an
  isotropic plate.
- **Not quasi-isotropic in bending**: in $D$ the trigonometric terms are
  weighted with $z^2$. The +45 degree plies are farther from the mid-surface
  than the -45 degree plies, so $\xi_{D2} \neq 0$ and
  $D_{16} = D_{26} = h^3 u_2 \xi_{D2}/24 \neq 0$ (the $\sin 4\theta$ terms
  vanish at $\pm 45$ degrees).
- **Equivalent properties**: with $a = [[A, B], [B, D]]^{-1}$,
  $E_1 = 1/(h a_{11})$, $E_2 = 1/(h a_{22})$, $G_{12} = 1/(h a_{66})$ and
  $\nu_{12} = -a_{12}/a_{11}$. Since $B = 0$, the membrane block of $a$ is
  $A^{-1}$, and the isotropic form of $A$ gives $E_1 = E_2$ and
  $G_{12} = E_1/(2(1 + \nu_{12}))$.
- **Transverse shear**: `Ats` is the corrected stiffness of the FSDT, by
  default with the equilibrium approach of Rohwer [1], see
  [Shear correction](../shear_correction.rst). It is symmetric, and the
  informative factors are the ratios to the constant-strain stiffness
  $\bar{A}_{ts}$: `scf_k13 = A55/Abar55` and `scf_k23 = A44/Abar44`.
- **Mass per unit area**: $I_0 = \sum_k \int \rho^{(k)} dz = \rho h$, see
  [Mass integrals](../mass_integrals.rst).

In [1]:
import sys
sys.path.insert(0, '../tests') # helpers of the tests

import numpy as np

from composites.utils import (read_laminaprop, laminated_plate,
        isotropic_plate)
from composites.core import (laminate_from_LaminationParameters,
                             make_balanced_LP, make_orthotropic_LP,
                             make_symmetric_LP)

In [2]:
# quasi-isotropic CFRP laminate [0/45/-45/90]s, SI units
laminaprop = (142e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 3.2e9) # E11, E22, nu12, G12, G13, G23
stack = [0, 45, -45, 90, 90, -45, 45, 0] # from the bottom to the top
lam = laminated_plate(stack, plyt=0.125e-3, laminaprop=laminaprop,
                      rho=1600.)
# ABD relates {Nxx, Nyy, Nxy, Mxx, Myy, Mxy} to the generalized strains
# {exx0, eyy0, gxy0, kxx, kyy, kxy}
ABD = lam.ABD
assert ABD.shape == (6, 6)
assert np.allclose(ABD[:3, :3], lam.A) and np.allclose(ABD[:3, 3:], lam.B)
assert np.allclose(ABD[3:, 3:], lam.D)
np.set_printoptions(precision=4)
print(ABD)

[[ 5.9947e+07  1.8214e+07  0.0000e+00  0.0000e+00  8.5265e-14  0.0000e+00]
 [ 1.8214e+07  5.9947e+07  1.8626e-09  8.5265e-14  0.0000e+00  0.0000e+00]
 [ 0.0000e+00  1.8626e-09  2.0866e+07  0.0000e+00  0.0000e+00 -2.8422e-13]
 [ 0.0000e+00  8.5265e-14  0.0000e+00  8.3812e+00  1.2715e+00  5.2322e-01]
 [ 8.5265e-14  0.0000e+00  0.0000e+00  1.2715e+00  2.1027e+00  5.2322e-01]
 [ 0.0000e+00  0.0000e+00 -2.8422e-13  5.2322e-01  5.2322e-01  1.4925e+00]]


In [3]:
# symmetric laminate: no membrane-bending coupling
assert np.allclose(lam.B, 0, atol=1e-12*lam.A11*lam.h)
# quasi-isotropic membrane stiffness
assert np.isclose(lam.A11, lam.A22, rtol=1e-12)
assert np.isclose(lam.A66, (lam.A11 - lam.A12)/2, rtol=1e-12)
assert np.allclose([lam.A16, lam.A26], 0, atol=1e-12*lam.A11)
# but not quasi-isotropic in bending: the +-45 plies give D16, D26
assert abs(lam.D16) > 1e-3*lam.D11
print('A11, A22, A66 =', lam.A11, lam.A22, lam.A66)
print('D11, D16, D26 =', lam.D11, lam.D16, lam.D26)

A11, A22, A66 = 59947200.58149737 59947200.58149738 20866481.02377958
D11, D16, D26 = 8.381249306528147 0.5232163320122458 0.5232163320122458


In [4]:
# equivalent membrane properties
assert np.isclose(lam.e1, lam.e2, rtol=1e-12)
assert np.isclose(lam.g12, lam.e1/(2*(1 + lam.nu12)), rtol=1e-12)
# FSDT: transverse shear stiffness with the shear correction applied
assert lam.shear_correction == 'rohwer'
assert np.allclose(lam.Ats, lam.Ats.T)
assert np.isclose(lam.scf_k13, lam.A55/lam.Abar55, rtol=1e-15)
assert np.isclose(lam.scf_k23, lam.A44/lam.Abar44, rtol=1e-15)
# mass per unit area
assert np.isclose(lam.intrho, 1600.*lam.h, rtol=1e-12)
print('e1, e2, g12, nu12 =', lam.e1, lam.e2, lam.g12, lam.nu12)
print('Ats =\n', lam.Ats)
print('scf_k13, scf_k23 =', lam.scf_k13, lam.scf_k23)
print('intrho =', lam.intrho)

e1, e2, g12, nu12 = 54413022468.87343 54413022468.87343 20866481023.77958 0.3038380167423538
Ats =
 [[2948931.4831   43511.7955]
 [  43511.7955 3247625.5097]]
scf_k13, scf_k23 = 0.7825603637815084 0.7105858995323155
intrho = 1.5999999999999999


## Comparison with reference values

The test `test_laminated_plate_plane_stress` of `tests/test_core.py` checks all
the matrices of the CLPT, FSDT and TSDT of the unsymmetric laminate $[0/45/90]$,
plies of 0.125 mm of a material with $E_1 = 71$ GPa, $E_2 = 7$ GPa,
$\nu_{12} = 0.28$ and $G_{12} = G_{13} = G_{23} = 7$ GPa:

- $A$, $B$, $D$, $E$, $F$, $H$ against stored values; no publication is given
  as their source. The transverse shear matrices of the TSDT,
  $(\bar{A}_{ts}, D_{ts}, F_{ts}) = \sum_k \int C_s^{(k)} (1, z^2, z^4) \, dz$,
  can be checked by hand: with $G_{13} = G_{23} = G$ the transverse shear
  stiffness of every ply is $C_s^{(k)} = G I$ for any angle, and
  $\bar{A}_{ts} = G h I$, $D_{ts} = G h^3/12 \, I$ and $F_{ts} = G h^5/80 \, I$,
  i.e. 2625000, 0.03076172 and 6.4888e-10 for $h = 0.375$ mm, the stored
  values.
- `Ats` against Eq. (43) of Rohwer [1], computed by the independent NumPy
  implementation `RohwerReference` of `tests/test_transverse_shear.py`. In the
  equilibrium approach the transverse shear stresses of two cylindrical
  bending states, $M_{xx,x} = Q_x$, $M_{xy,x} = Q_y$ and $M_{yy,y} = Q_y$,
  $M_{xy,y} = Q_x$, are integrated from the equilibrium equations,
  $\tau_{xz,z} = -\sigma_{xx,x}$ and $\tau_{yz,z} = -\sigma_{yy,y}$, which
  gives a distribution $\{\tau_{yz}, \tau_{xz}\} = f^{(k)}(z) \, Q$ quadratic
  within each ply, and the stiffness follows from the complementary energy:

$$
A_{ts} = \left[ \sum_k \int_{z_k}^{z_{k+1}} f^{(k)T}
\left(C_s^{(k)}\right)^{-1} f^{(k)} \, dz \right]^{-1}
$$

The integrand is of degree 4 in $z$, which the 3-point Gauss-Legendre rule per
ply of `RohwerReference` integrates exactly. The two implementations solve the
same equations, so they agree to round-off (`rtol=1e-12`). Since
$G_{13} = G_{23}$, $\bar{A}_{45} = 0$; the coupling $A_{45} \neq 0$ of the
equilibrium approach comes from the in-plane couplings of the 45 degree ply,
e.g. the bending-twisting terms $D_{16}$, $D_{26}$.

In [5]:
import pytest
from test_transverse_shear import RohwerReference, stack_from_laminate

lamprop = (71e9, 7e9, 0.28, 7e9, 7e9, 7e9)
stack = [0, 45, 90]
plyt = 0.000125
lam = laminated_plate(stack, plyt, lamprop)
A = np.array([[ 13280892.30559593, 2198758.85719477, 2015579.57848837],
              [  2198758.85719477,13280892.30559593, 2015579.57848837],
              [  2015579.57848837, 2015579.57848837, 4083033.36210029]])
B = np.array([[ -1.00778979e+03, 0.00000000e+00, 8.53496487e-15],
              [  0.00000000e+00, 1.00778979e+03, 5.31743621e-14],
              [  8.53496487e-15, 5.31743621e-14, 0.00000000e+00]])
D = np.array([[ 0.1708233 , 0.01057886, 0.00262445],
              [ 0.01057886, 0.1708233 , 0.00262445],
              [ 0.00262445, 0.00262445, 0.0326602 ]])
E = np.array([[-1.96833943e-05, 0.00000000e+00, 1.66698533e-22],
              [0.00000000e+00, 1.96833943e-05, 1.03856176e-21],
              [1.66698533e-22, 1.03856176e-21, 0.00000000e+00]])
F = np.array([[3.63890059e-09, 1.87551265e-10, 6.15106073e-12],
              [1.87551265e-10, 3.63890059e-09, 6.15106073e-12],
              [6.15106073e-12, 6.15106073e-12, 6.53329570e-10]])
H = np.array([[9.14779627e-17, 4.61039304e-18, 1.71625578e-20],
              [4.61039304e-18, 9.14779627e-17, 1.71625578e-20],
              [1.71625578e-20, 1.71625578e-20, 1.63068348e-17]])
Abar_ts = np.array([[ 2625000.,       0.],
                    [       0., 2625000.]])
Ats = RohwerReference(stack_from_laminate(lam)).Ats
Dts = np.array([[0.03076172, 0.],
                [0., 0.03076172]])
Fts = np.array([[6.48880005e-10, 0.00000000e+00],
                [0.00000000e+00, 6.48880005e-10]])

assert np.allclose(lam.A, A)
assert np.allclose(lam.B, B)
assert np.allclose(lam.D, D)
assert np.allclose(lam.E, E)
assert np.allclose(lam.F, F)
assert np.allclose(lam.H, H)
assert np.allclose(lam.Abar_ts, Abar_ts)
assert np.allclose(lam.Ats, Ats, rtol=1e-12, atol=1e-14*Ats.max())
assert np.allclose(lam.Dts, Dts)
assert np.allclose(lam.Fts, Fts)
print('Ats =\n', lam.Ats)
print('Ats of Rohwer (1988), Eq. (43) =\n', Ats)

Ats =
 [[2014905.2727  -27210.2938]
 [ -27210.2938 2014905.2727]]
Ats of Rohwer (1988), Eq. (43) =
 [[2014905.2727  -27210.2938]
 [ -27210.2938 2014905.2727]]


## Lamination parameters and special laminates

For a laminate of a single material the stiffnesses $A$, $B$, $D$ are linear
in the lamination parameters, with the invariants as coefficients
([Lamination parameters](../lamination_parameters.rst); Jones [2], Eq.
(4.24)). `laminate_from_LaminationParameters` with the parameters of
`calc_lamination_parameters` therefore gives back exactly the $A$, $B$ and $D$
of `laminated_plate`. A laminate created from lamination parameters has no
plies, hence no shear correction: its `Ats` is the constant-strain
$\bar{A}_{ts}$ of the original laminate.

`make_balanced`, `make_orthotropic` and `make_symmetric` set to zero the
integrals that vanish in a balanced, orthotropic or symmetric laminate, and
`make_balanced_LP`, `make_orthotropic_LP` and `make_symmetric_LP` set to zero
the corresponding lamination parameters, so that both paths give the same
matrices, see [Laminate stiffness and plate theories](../constitutive_matrices.rst).
`make_smeared` is the limit of many repetitions of the stacking sequence:
$B = 0$ and $D = h^2 A/12$, the stiffnesses of a homogeneous plate.

In [6]:
with pytest.warns(DeprecationWarning):
    lam.calc_scf()
lam.calc_equivalent_properties()
lp = lam.calc_lamination_parameters()
matlamina = lam.plies[0].matlamina
thickness = lam.h
lam_2 = laminate_from_LaminationParameters(thickness, matlamina, lp)
assert np.allclose(lam_2.A, lam.A)
assert np.allclose(lam_2.B, lam.B)
assert np.allclose(lam_2.D, lam.D)
assert np.allclose(lam_2.Ats, lam.Abar_ts)

lam.make_balanced()
make_balanced_LP(lp)
lam_2 = laminate_from_LaminationParameters(thickness, matlamina, lp)
assert np.allclose(lam_2.A, lam.A)
assert np.allclose(lam_2.B, lam.B)
assert np.allclose(lam_2.D, lam.D)
assert np.allclose(lam_2.Ats, lam.Abar_ts)

lam.make_orthotropic()
make_orthotropic_LP(lp)
lam_2 = laminate_from_LaminationParameters(thickness, matlamina, lp)
assert np.allclose(lam_2.A, lam.A)
assert np.allclose(lam_2.B, lam.B)
assert np.allclose(lam_2.D, lam.D)
assert np.allclose(lam_2.Ats, lam.Abar_ts)

lam = laminated_plate(stack, plyt, lamprop)
lp = lam.calc_lamination_parameters()
lam.make_symmetric()
make_symmetric_LP(lp)
lam_2 = laminate_from_LaminationParameters(thickness, matlamina, lp)
assert np.allclose(lam_2.A, lam.A)
assert np.allclose(lam_2.B, lam.B)
assert np.allclose(lam_2.D, lam.D)
assert np.allclose(lam_2.Ats, lam.Abar_ts)

lam = laminated_plate(stack, plyt, lamprop)
lp = lam.calc_lamination_parameters()
lam.make_smeared()
assert np.allclose(lam.D, lam.h**2/12*lam.A)
assert np.allclose(lam.B, 0)
print('smeared laminate, D =\n', lam.D)

smeared laminate, D =
 [[0.1556 0.0258 0.0236]
 [0.0258 0.1556 0.0236]
 [0.0236 0.0236 0.0478]]


## References

1. Rohwer, K., Improved Transverse Shear Stiffnesses for Layered Finite Elements, DFVLR-FB 88-32, 1988.
2. Jones, R. M., Mechanics of Composite Materials, Second Edition, Taylor & Francis, Philadelphia, 1999.